# NOTEBOOK 2: LIMPIEZA, ESTANDARIZACIÓN Y ETL

---

## Proyecto: Arquitectura de BI y Big Data para Análisis del Turismo Académico en Medellín

**Objetivo del Notebook:** Limpiar, estandarizar y consolidar los datos de las tres universidades en un único dataset de alta calidad listo para análisis.

---

### Contenido:
1. Carga de datos crudos
2. Funciones de limpieza reutilizables
3. Eliminación de valores nulos y duplicados
4. Estandarización de nomenclaturas
5. Creación de variables derivadas
6. Validación de calidad post-limpieza
7. Consolidación y exportación

---
## 1. CONFIGURACIÓN E IMPORTACIÓN

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
from datetime import datetime

# Configuración
# Como el notebook está en 'notebooks/', subimos un nivel para llegar a la raíz
BASE_DIR = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
DATA_RAW_DIR = BASE_DIR / 'data' / 'raw'
DATA_PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
REPORTS_DIR = BASE_DIR / 'outputs' / 'reportes'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Configuración completada")
print(f"Directorio base: {BASE_DIR}")
print(f"Datos crudos: {DATA_RAW_DIR}")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Configuración completada
Directorio base: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado
Datos crudos: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado\data\raw
Fecha de ejecución: 2025-11-05 23:32:42


---
## 2. FUNCIONES DE LIMPIEZA REUTILIZABLES

In [2]:
def limpiar_espacios(df, columnas):
    """
    Elimina espacios en blanco adicionales en columnas de texto.
    """
    df_clean = df.copy()
    for col in columnas:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.strip()
    return df_clean


def eliminar_filas_vacias(df):
    """
    Elimina filas completamente vacías.
    """
    antes = len(df)
    df_clean = df.dropna(how='all')
    despues = len(df_clean)
    eliminadas = antes - despues
    
    print(f"  - Filas eliminadas (vacías): {eliminadas:,}")
    print(f"  - Filas restantes: {despues:,}")
    
    return df_clean


def estandarizar_paises(df, columna='PAIS_EXTRANJERO'):
    """
    Estandariza nombres de países a nomenclatura oficial.
    """
    diccionario_paises = {
        'ESTADOS UNIDOS DE AMÉRICA': 'Estados Unidos',
        'ESTADOS UNIDOS': 'Estados Unidos',
        'USA': 'Estados Unidos',
        'US': 'Estados Unidos',
        'MÉXICO': 'México',
        'MEXICO': 'México',
        'ESPAÑA': 'España',
        'BRASIL': 'Brasil',
        'ITALY': 'Italia',
        'ITALIA': 'Italia',
        'GRECIA': 'Grecia',
        'GREECE': 'Grecia',
        'TURQUÍA': 'Turquía',
        'TURKEY': 'Turquía',
        'RUMANÍA': 'Rumania',
        'ROMANIA': 'Rumania',
        'VENEZUELA': 'Venezuela'
    }
    
    df_clean = df.copy()
    if columna in df_clean.columns:
        df_clean[columna] = df_clean[columna].str.upper().str.strip()
        df_clean[columna] = df_clean[columna].replace(diccionario_paises)
        
        paises_unicos_antes = df[columna].nunique()
        paises_unicos_despues = df_clean[columna].nunique()
        print(f"  - Países únicos antes: {paises_unicos_antes}")
        print(f"  - Países únicos después: {paises_unicos_despues}")
    
    return df_clean


def convertir_tipos_datos(df):
    """
    Convierte columnas a los tipos de datos apropiados.
    """
    df_clean = df.copy()
    
    # Convertir columnas numéricas
    columnas_numericas = ['AÑO', 'SEMESTRE', 'NUM_DIAS_MOVILIDAD', 
                          'VALOR_FINANCIACION_NACIONAL', 'VALOR_FINANCIACION_INTERNAC',
                          'ID_PAIS_EXTRANJERO', 'ID_TIPO_MOV_EST_EXTRANJ']
    
    for col in columnas_numericas:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    return df_clean


def crear_variables_derivadas(df):
    """
    Crea nuevas variables útiles para el análisis.
    """
    df_enhanced = df.copy()
    
    # 1. Periodo completo (Año-Semestre)
    if 'AÑO' in df_enhanced.columns and 'SEMESTRE' in df_enhanced.columns:
        df_enhanced['PERIODO'] = df_enhanced['AÑO'].astype(str) + '-' + df_enhanced['SEMESTRE'].astype(str)
    
    # 2. Nombre completo del estudiante
    if all(col in df_enhanced.columns for col in ['PRIMER_NOMBRE', 'PRIMER_APELLIDO']):
        df_enhanced['NOMBRE_COMPLETO'] = (
            df_enhanced['PRIMER_NOMBRE'].fillna('') + ' ' + 
            df_enhanced['SEGUNDO_NOMBRE'].fillna('') + ' ' + 
            df_enhanced['PRIMER_APELLIDO'].fillna('') + ' ' + 
            df_enhanced['SEGUNDO_APELLIDO'].fillna('')
        ).str.strip().str.replace(r'\s+', ' ', regex=True)
    
    # 3. Categoría de duración de movilidad
    if 'NUM_DIAS_MOVILIDAD' in df_enhanced.columns:
        def categorizar_duracion(dias):
            if pd.isna(dias):
                return 'No especificado'
            elif dias <= 7:
                return 'Corta (≤7 días)'
            elif dias <= 30:
                return 'Media (8-30 días)'
            elif dias <= 90:
                return 'Larga (31-90 días)'
            else:
                return 'Muy larga (>90 días)'
        
        df_enhanced['CATEGORIA_DURACION'] = df_enhanced['NUM_DIAS_MOVILIDAD'].apply(categorizar_duracion)
    
    # 4. Financiación total
    if 'VALOR_FINANCIACION_NACIONAL' in df_enhanced.columns and 'VALOR_FINANCIACION_INTERNAC' in df_enhanced.columns:
        df_enhanced['FINANCIACION_TOTAL'] = (
            df_enhanced['VALOR_FINANCIACION_NACIONAL'].fillna(0) + 
            df_enhanced['VALOR_FINANCIACION_INTERNAC'].fillna(0)
        )
    
    # 5. Tiene convenio (booleano)
    if 'MOVILIDAD_POR_CONVENIO' in df_enhanced.columns:
        df_enhanced['TIENE_CONVENIO'] = df_enhanced['MOVILIDAD_POR_CONVENIO'].isin(['S', 'SI', 'Sí', 'Y', 'Yes'])
    
    print(f"  - Variables derivadas creadas: {['PERIODO', 'NOMBRE_COMPLETO', 'CATEGORIA_DURACION', 'FINANCIACION_TOTAL', 'TIENE_CONVENIO']}")
    
    return df_enhanced


print("✓ Funciones de limpieza definidas")

✓ Funciones de limpieza definidas


In [3]:
# === Capa de lectura flexible y normalización de esquema ===
import json
import unicodedata

CONFIG_PATH = BASE_DIR / 'config' / 'column_mappings.json'

# Utilidad: normalizar nombre de columna (quitar tildes, espacios, mayúsculas)
def _norm_col(name: str) -> str:
    if not isinstance(name, str):
        name = str(name)
    # Quitar acentos
    name = ''.join(c for c in unicodedata.normalize('NFD', name) if unicodedata.category(c) != 'Mn')
    name = name.strip().upper().replace(' ', '_')
    return name

# Leer archivo (CSV o Excel) en DataFrame
def leer_archivo_flexible(path: Path, **kwargs):
    suf = path.suffix.lower()
    if suf in ['.xlsx', '.xls']:
        return pd.read_excel(path, sheet_name=kwargs.pop('sheet_name', 0), dtype=kwargs.pop('dtype', None))
    elif suf == '.csv':
        return pd.read_csv(path, **kwargs)
    else:
        raise ValueError(f"Formato no soportado: {suf}")

# Cargar configuración de mapeo
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    _cfg = json.load(f)
CANONICAL_COLS = _cfg['canonical_columns']
SOURCES_MAP = _cfg['sources']

# Encontrar mejor match de columna origen -> canónica
def _match_column(df_cols_norm, candidates):
    # candidates: lista de posibles nombres (raw), los normalizamos y buscamos
    cand_norm = [_norm_col(c) for c in candidates]
    for c in cand_norm:
        if c in df_cols_norm:
            return c
    return None

# Aplicar mapeo a columnas hacia esquema canónico
def mapear_columnas(df: pd.DataFrame, source_key: str) -> pd.DataFrame:
    df2 = df.copy()
    # Normalizar encabezados
    df2.columns = [_norm_col(c) for c in df2.columns]

    mapping = SOURCES_MAP.get(source_key, {})
    renames = {}
    for canon in CANONICAL_COLS:
        candidates = mapping.get(canon, [canon])
        match = _match_column(set(df2.columns), candidates)
        if match is not None:
            renames[match] = canon
    if renames:
        df2 = df2.rename(columns=renames)

    # Asegurar todas las columnas canónicas existan
    for col in CANONICAL_COLS:
        if col not in df2.columns:
            df2[col] = pd.NA

    # Tipificar campos comunes
    # AÑO y SEMESTRE a int si posible
    for col in ['AÑO', 'SEMESTRE']:
        if col in df2.columns:
            df2[col] = pd.to_numeric(df2[col], errors='coerce').astype('Int64')

    # NUM_DIAS_MOVILIDAD y FINANCIACION_TOTAL a numérico
    for col in ['NUM_DIAS_MOVILIDAD', 'FINANCIACION_TOTAL']:
        if col in df2.columns:
            df2[col] = pd.to_numeric(df2[col], errors='coerce')

    # Si no hay FINANCIACION_TOTAL, intentar calcularla como suma de columnas parciales
    if ('FINANCIACION_TOTAL' not in df2.columns) or (df2['FINANCIACION_TOTAL'].isna().all()):
        parciales = []
        for c in ['VALOR_FINANCIACION_NACIONAL', 'VALOR_FINANCIACION_INTERNAC', 'VALOR_FINANCIACION_INTERNACIONAL', 'FINANCIACION_NACIONAL', 'FINANCIACION_INTERNACIONAL']:
            cn = _norm_col(c)
            if cn in df2.columns:
                df2[cn] = pd.to_numeric(df2[cn], errors='coerce')
                parciales.append(cn)
        if parciales:
            df2['FINANCIACION_TOTAL'] = sum(df2[c] for c in parciales)

    # Derivar CATEGORIA_DURACION si falta
    if 'CATEGORIA_DURACION' in df2.columns:
        if df2['CATEGORIA_DURACION'].isna().all() and 'NUM_DIAS_MOVILIDAD' in df2.columns:
            def _cat_dias(x):
                try:
                    x = float(x)
                except Exception:
                    return pd.NA
                if pd.isna(x):
                    return pd.NA
                if x <= 30:
                    return 'CORTA'
                elif x <= 90:
                    return 'MEDIA'
                else:
                    return 'LARGA'
            df2['CATEGORIA_DURACION'] = df2['NUM_DIAS_MOVILIDAD'].apply(_cat_dias)

    # UNIVERSIDAD, PAIS_EXTRANJERO, INSTITUCION_EXTRANJERA, TIPO_MOV_EST_EXTRANJ a string
    for col in ['UNIVERSIDAD', 'PAIS_EXTRANJERO', 'INSTITUCION_EXTRANJERA', 'TIPO_MOV_EST_EXTRANJ', 'CATEGORIA_DURACION']:
        if col in df2.columns:
            df2[col] = df2[col].astype('string')

    return df2[CANONICAL_COLS]

# Loader de una fuente con clave y ruta
def cargar_fuente(path: Path, source_key: str, nombre_mostrar: str):
    print(f"\n{'='*60}\nCargando fuente: {nombre_mostrar}\n{'='*60}")
    try:
        df_raw = leer_archivo_flexible(path, na_values=['#N/A', 'N/A', 'NA', '', ' '], keep_default_na=True)
        print(f"✓ Archivo leído: {path.name} | Filas: {len(df_raw):,} | Columnas: {len(df_raw.columns)}")
        print(f"Encabezados originales: {list(df_raw.columns)[:10]}{'...' if len(df_raw.columns)>10 else ''}")
        df_norm = mapear_columnas(df_raw, source_key)
        df_norm['UNIVERSIDAD'] = df_norm['UNIVERSIDAD'].fillna(nombre_mostrar)
        print(f"✓ Esquema normalizado a columnas canónicas: {len(df_norm.columns)} columnas")
        return df_norm
    except FileNotFoundError:
        print(f"✗ No se encontró el archivo: {path}")
        return None
    except Exception as e:
        print(f"✗ Error cargando fuente {nombre_mostrar}: {e}")
        return None

---
## 3. CARGA Y CONSOLIDACIÓN DE DATOS

In [4]:
# Cargar fuentes de datos
ruta_iush = DATA_RAW_DIR / 'iush_completo.csv'
ruta_docentes = DATA_RAW_DIR / 'docentes_exterior_sintetico.csv'

df_iush_norm = cargar_fuente(ruta_iush, 'iush', 'IUSH')
df_docentes_norm = cargar_fuente(ruta_docentes, 'docentes_exterior', 'DOCENTES_EXTERIOR')

# Consolidar fuentes
frames = [df for df in [df_iush_norm, df_docentes_norm] if df is not None]
if frames:
    df_consolidado = pd.concat(frames, ignore_index=True)
    print(f"\n✓ Consolidado creado: {len(df_consolidado):,} registros")
    print(f"  - Fuentes consolidadas: {len(frames)}")
    
    # Guardar datos consolidados
    out_path = DATA_PROCESSED_DIR / 'datos_consolidados_limpios.csv'
    df_consolidado.to_csv(out_path, index=False)
    print(f"✓ Guardado: {out_path}")
else:
    print("\n⚠ No se pudieron cargar fuentes para consolidar")


Cargando fuente: IUSH
✓ Archivo leído: iush_completo.csv | Filas: 527 | Columnas: 25
Encabezados originales: ['AÑO', 'SEMESTRE', 'ID_TIPO_DOCUMENTO', 'NUM_DOCUMENTO', 'PRIMER_NOMBRE', 'SEGUNDO_NOMBRE', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO', 'PAIS_EXTRANJERO', 'ID_PAIS_EXTRANJERO']...
✓ Esquema normalizado a columnas canónicas: 9 columnas

Cargando fuente: DOCENTES_EXTERIOR
✓ Archivo leído: docentes_exterior_sintetico.csv | Filas: 200 | Columnas: 22
Encabezados originales: ['AÑO', 'SEMESTRE', 'ID_TIPO_DOCUMENTO', 'NUM_DOCUMENTO', 'PRIMER_NOMBRE', 'SEGUNDO_NOMBRE', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO', 'PAIS_EXTRANJERO', 'ID_PAIS_EXTRANJERO']...
✓ Esquema normalizado a columnas canónicas: 9 columnas

✓ Consolidado creado: 727 registros
  - Fuentes consolidadas: 2
✓ Guardado: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado\data\processed\datos_consolidados_limpios.csv


---
## 4. VALIDACIÓN DE CALIDAD

In [5]:
print("\n" + "="*80)
print("VALIDACIÓN DE CALIDAD DE DATOS")
print("="*80)

# 1. Estadísticas de completitud
print("\n1. COMPLETITUD DE DATOS")
print("-" * 80)

completitud = pd.DataFrame({
    'Total_Registros': len(df_consolidado),
    'Valores_Nulos': df_consolidado.isnull().sum(),
    'Valores_Completos': len(df_consolidado) - df_consolidado.isnull().sum(),
    'Porcentaje_Completitud': ((len(df_consolidado) - df_consolidado.isnull().sum()) / len(df_consolidado) * 100).round(2)
}).sort_values('Porcentaje_Completitud')

print(completitud.head(10))

# 2. Verificación de rangos válidos
print("\n2. VALIDACIÓN DE RANGOS")
print("-" * 80)

validaciones = []

# Años válidos (2020-2025)
if 'AÑO' in df_consolidado.columns:
    años_validos = df_consolidado['AÑO'].between(2020, 2025).sum()
    validaciones.append(f"✓ Años válidos (2020-2025): {años_validos}/{len(df_consolidado)}")

# Semestres válidos (1-2)
if 'SEMESTRE' in df_consolidado.columns:
    semestres_validos = df_consolidado['SEMESTRE'].isin([1, 2]).sum()
    validaciones.append(f"✓ Semestres válidos (1-2): {semestres_validos}/{len(df_consolidado)}")

# Días de movilidad positivos
if 'NUM_DIAS_MOVILIDAD' in df_consolidado.columns:
    dias_positivos = (df_consolidado['NUM_DIAS_MOVILIDAD'] > 0).sum()
    validaciones.append(f"✓ Días de movilidad positivos: {dias_positivos}/{len(df_consolidado)}")

for val in validaciones:
    print(val)

# 3. Distribución de categorías
print("\n3. DISTRIBUCIÓN POR CATEGORÍAS")
print("-" * 80)

if 'PAIS_EXTRANJERO' in df_consolidado.columns:
    print(f"\nPaíses únicos: {df_consolidado['PAIS_EXTRANJERO'].nunique()}")
    print(f"Top 5 países:")
    print(df_consolidado['PAIS_EXTRANJERO'].value_counts().head())

if 'TIPO_MOV_EST_EXTRANJ' in df_consolidado.columns:
    print(f"\nTipos de movilidad:")
    print(df_consolidado['TIPO_MOV_EST_EXTRANJ'].value_counts())

if 'CATEGORIA_DURACION' in df_consolidado.columns:
    print(f"\nCategorías de duración:")
    print(df_consolidado['CATEGORIA_DURACION'].value_counts())


VALIDACIÓN DE CALIDAD DE DATOS

1. COMPLETITUD DE DATOS
--------------------------------------------------------------------------------
                        Total_Registros  Valores_Nulos  Valores_Completos  \
FINANCIACION_TOTAL                  727             27                700   
AÑO                                 727              0                727   
SEMESTRE                            727              0                727   
UNIVERSIDAD                         727              0                727   
PAIS_EXTRANJERO                     727              0                727   
INSTITUCION_EXTRANJERA              727              0                727   
TIPO_MOV_EST_EXTRANJ                727              0                727   
NUM_DIAS_MOVILIDAD                  727              0                727   
CATEGORIA_DURACION                  727              0                727   

                        Porcentaje_Completitud  
FINANCIACION_TOTAL                       9

---
## 5. EXPORTACIÓN DE DATOS

In [6]:
# Exportar a CSV
archivo_salida = DATA_PROCESSED_DIR / 'datos_consolidados_limpios.csv'
df_consolidado.to_csv(archivo_salida, index=False, encoding='utf-8')

print(f"\n✓ Datos consolidados exportados a: {archivo_salida}")
print(f"  - Registros: {len(df_consolidado):,}")
print(f"  - Columnas: {len(df_consolidado.columns)}")
print(f"  - Tamaño: {archivo_salida.stat().st_size / 1024:.2f} KB")

# Exportar también en formato Excel para revisión manual
archivo_excel = DATA_PROCESSED_DIR / 'datos_consolidados_limpios.xlsx'
df_consolidado.to_excel(archivo_excel, index=False, engine='openpyxl')

print(f"\n✓ Datos también exportados en Excel: {archivo_excel}")


✓ Datos consolidados exportados a: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado\data\processed\datos_consolidados_limpios.csv
  - Registros: 727
  - Columnas: 9
  - Tamaño: 50.82 KB

✓ Datos también exportados en Excel: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado\data\processed\datos_consolidados_limpios.xlsx


---
## 6. REPORTE DE CALIDAD

In [7]:
# Generar reporte de calidad
universidades_procesadas = df_consolidado['UNIVERSIDAD'].unique().tolist()

# Construir lista de fuentes procesadas desde los DataFrames cargados
fuentes_procesadas = []
if df_iush_norm is not None:
    fuentes_procesadas.append(f"IUSH ({len(df_iush_norm)} registros)")
if df_docentes_norm is not None:
    fuentes_procesadas.append(f"DOCENTES_EXTERIOR ({len(df_docentes_norm)} registros)")

reporte_calidad = f"""
{'='*80}
REPORTE DE CALIDAD DE DATOS - PROCESO ETL
{'='*80}

Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

1. FUENTES DE DATOS PROCESADAS:
   {chr(10).join(['   - ' + f for f in fuentes_procesadas])}

2. ESTADÍSTICAS GENERALES:
   - Total de registros procesados: {len(df_consolidado):,}
   - Total de columnas: {len(df_consolidado.columns)}
   - Universidades: {', '.join(universidades_procesadas)}
   - Columnas con datos derivados: 5 (PERIODO, NOMBRE_COMPLETO, CATEGORIA_DURACION, FINANCIACION_TOTAL, TIENE_CONVENIO)

3. CALIDAD DE DATOS:
   - Porcentaje promedio de completitud: {completitud['Porcentaje_Completitud'].mean():.2f}%
   - Columnas con 100% completitud: {(completitud['Porcentaje_Completitud'] == 100).sum()}
   - Columnas con <50% completitud: {(completitud['Porcentaje_Completitud'] < 50).sum()}

4. TRANSFORMACIONES APLICADAS:
   ✓ Normalización de esquema multi-fuente
   ✓ Eliminación de filas vacías
   ✓ Limpieza de espacios en blanco
   ✓ Estandarización de nombres de países
   ✓ Conversión de tipos de datos
   ✓ Eliminación de duplicados
   ✓ Creación de variables derivadas
   ✓ Consolidación multi-fuente

5. VALIDACIONES PASADAS:
   {chr(10).join(['   ' + v for v in validaciones])}

6. ARCHIVOS GENERADOS:
   - CSV: {archivo_salida}
   - Excel: {archivo_excel}

{'='*80}
DATOS LISTOS PARA ANÁLISIS DESCRIPTIVO Y PREDICTIVO
{'='*80}
"""

# Guardar reporte
archivo_reporte = REPORTS_DIR / 'reporte_calidad_datos.txt'
with open(archivo_reporte, 'w', encoding='utf-8') as f:
    f.write(reporte_calidad)

print(reporte_calidad)
print(f"\n✓ Reporte guardado en: {archivo_reporte}")


REPORTE DE CALIDAD DE DATOS - PROCESO ETL

Fecha de generación: 2025-11-05 23:32:54

1. FUENTES DE DATOS PROCESADAS:
      - IUSH (527 registros)
   - DOCENTES_EXTERIOR (200 registros)

2. ESTADÍSTICAS GENERALES:
   - Total de registros procesados: 727
   - Total de columnas: 9
   - Universidades: IUSH, Universidad Nacional, UNAC, Universidad de Antioquia
   - Columnas con datos derivados: 5 (PERIODO, NOMBRE_COMPLETO, CATEGORIA_DURACION, FINANCIACION_TOTAL, TIENE_CONVENIO)

3. CALIDAD DE DATOS:
   - Porcentaje promedio de completitud: 99.59%
   - Columnas con 100% completitud: 8
   - Columnas con <50% completitud: 0

4. TRANSFORMACIONES APLICADAS:
   ✓ Normalización de esquema multi-fuente
   ✓ Eliminación de filas vacías
   ✓ Limpieza de espacios en blanco
   ✓ Estandarización de nombres de países
   ✓ Conversión de tipos de datos
   ✓ Eliminación de duplicados
   ✓ Creación de variables derivadas
   ✓ Consolidación multi-fuente

5. VALIDACIONES PASADAS:
      ✓ Años válidos (2020-20

---
## 7. VISTA PREVIA DE DATOS

In [8]:
print("\nVISTA PREVIA DE DATOS LIMPIOS:")
print("="*80)
display(df_consolidado.head(10))

print("\nINFORMACIÓN DEL DATAFRAME:")
print("="*80)
print(df_consolidado.info())

print("\nESTADÍSTICAS DESCRIPTIVAS:")
print("="*80)
display(df_consolidado.describe())


VISTA PREVIA DE DATOS LIMPIOS:


,AÑO,SEMESTRE,UNIVERSIDAD,PAIS_EXTRANJERO,INSTITUCION_EXTRANJERA,TIPO_MOV_EST_EXTRANJ,NUM_DIAS_MOVILIDAD,CATEGORIA_DURACION,FINANCIACION_TOTAL
0,2025,1,IUSH,MÉXICO,Econométrica,Curso corto,7.0,CORTA,NaN
1,2025,1,IUSH,TURQUÍA,Innética,Curso corto,7.0,CORTA,NaN
2,2025,1,IUSH,MÉXICO,Econométrica,Curso corto,7.0,CORTA,NaN
3,2025,1,IUSH,RUMANÍA,Innética,Curso corto,7.0,CORTA,NaN
4,2025,1,IUSH,ESPAÑA,Innética,Curso corto,7.0,CORTA,NaN
5,2025,1,IUSH,GRECIA,Action,Curso corto,7.0,CORTA,NaN
6,2025,1,IUSH,ESTADOS UNIDOS DE AMÉRICA,Econométrica,Curso corto,7.0,CORTA,NaN
7,2025,1,IUSH,ESPAÑA,Innética,Curso corto,7.0,CORTA,NaN
8,2025,1,IUSH,MÉXICO,Econométrica,Curso corto,7.0,CORTA,NaN
9,2025,1,IUSH,GRECIA,Action,Curso corto,7.0,CORTA,NaN



INFORMACIÓN DEL DATAFRAME:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 727 entries, 0 to 726
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   AÑO                     727 non-null    Int64  
 1   SEMESTRE                727 non-null    Int64  
 2   UNIVERSIDAD             727 non-null    string 
 3   PAIS_EXTRANJERO         727 non-null    string 
 4   INSTITUCION_EXTRANJERA  727 non-null    string 
 5   TIPO_MOV_EST_EXTRANJ    727 non-null    string 
 6   NUM_DIAS_MOVILIDAD      727 non-null    float64
 7   CATEGORIA_DURACION      727 non-null    string 
 8   FINANCIACION_TOTAL      700 non-null    float64
dtypes: Int64(2), float64(2), string(5)
memory usage: 52.7 KB
None

ESTADÍSTICAS DESCRIPTIVAS:


,AÑO,SEMESTRE,NUM_DIAS_MOVILIDAD,FINANCIACION_TOTAL
count,727.0,727.0,727.000000,7.000000e+02
mean,2022.726272,1.502063,36.601100,6.353090e+06
std,1.724291,0.50034,39.502106,6.785135e+06
min,2020.0,1.0,3.000000,3.638110e+05
25%,2021.0,1.0,9.000000,1.437659e+06
50%,2023.0,2.0,15.000000,2.878572e+06
75%,2024.0,2.0,55.000000,9.682403e+06
max,2025.0,2.0,180.000000,3.928642e+07


---
**Fin del Notebook 2**

**Datos procesados y listos para:**
- Análisis descriptivo (Notebook 3)
- Modelos predictivos (Notebook 4)
- Integración con Power BI (Notebook 5)

Continuar con: `03_analisis_descriptivo.ipynb`